**Preliminari**

In [1]:
import sys
import os
from dotenv import load_dotenv  
from pathlib import Path

# if notebook is in PRIN/notebooks, parent() is PRIN
project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))
print("Added to sys.path:", project_root)

import json
from utils.schema_json import ReportData, AnnotatedReport
import time
from IPython.display import clear_output

from huggingface_hub import login

import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset, Dataset, DatasetDict

Added to sys.path: C:\Users\lucat\PythonRepositories\PRIN


**Impostiamo il device, scheda video se disponibile**

In [2]:
print(f'{torch.cuda.is_available() = }')  # True se la GPU è disponibile
print(f'{torch.cuda.device_count() = }')  # Numero di GPU disponibili
print(f'{torch.cuda.get_device_name(0) = }')  # Nome della GPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'{device = }')

torch.cuda.is_available() = True
torch.cuda.device_count() = 1
torch.cuda.get_device_name(0) = 'NVIDIA GeForce GTX 1060 6GB'
device = device(type='cuda')


**Huggingface login**

In [3]:
# Set the API key for HuggingFace
load_dotenv()  # Load environment variables from .env file
hf_api_key = os.getenv("HF_TOKEN")
login(token=hf_api_key)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


**Parametri**

In [4]:
# Parameters
TRAIN_FILE_NAME = "data_finetune_guido_openai_train.jsonl"
VALIDATION_FILE_NAME = "data_finetune_guido_openai_val.jsonl"
#FILE_NAMES = (TRAIN_FILE_NAME, VALIDATION_FILE_NAME)
TIPO = 'openai'

CHECKPOINT = "bert-base-multilingual-cased"

# Fields we don't want to predict
EXCLUDED_FIELDS = (
    'ore_inizio',
    'ore_fine',
    'spessore_parietale',
    'estensione_cranio_caudale',
    'distanza_oai',
    'linfonodi_sospetti',
    'numero_depositi'
)

**Load data**

In [5]:
# Carichiamo i nostri file JSON
file_names = {
    'train': TRAIN_FILE_NAME,
    'validation': VALIDATION_FILE_NAME,
}

paths = {
    split: Path('../data/ft-dataset/' + file_name) for split, file_name in file_names.items()
}

data = dict()
for split, path in paths.items():
    with open(path, "r", encoding="utf-8") as f:
        data_list = [json.loads(line) for line in f]
        data[split] = data_list

train_data, validation_data = data['train'], data['validation']

print(f"{len(train_data) = }")
print(f"{type(train_data[0]) = }")
print(f"{type(train_data[0]['messages'][1]['content']) = }")  # Report text
print(f"{type(train_data[0]['messages'][2]['content']) = }")  # Annotations

len(train_data) = 116
type(train_data[0]) = <class 'dict'>
type(train_data[0]['messages'][1]['content']) = <class 'str'>
type(train_data[0]['messages'][2]['content']) = <class 'str'>


In [6]:
annotated_reports: dict[str, list[AnnotatedReport]] = {split: [] for split in file_names.keys()}
for split in annotated_reports:
    for record in data[split]:
        report_text = record['messages'][1]['content'].lower()  # Tutte lettere minuscole
        if TIPO == 'openai':
            report_data = ReportData.model_validate_json(record['messages'][2]['content'])
        else:
            report_data = ReportData.model_validate(record['messages'][2]['content'])
        annotated_reports[split].append(AnnotatedReport(report_text=report_text, report_data=report_data))

**Load model and tokenizer**

In [7]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModel.from_pretrained(CHECKPOINT).to(device)

In [8]:
print(f"{tokenizer.pad_token_type_id = }")

tokenizer.pad_token_type_id = 0


In [9]:
# Check the maximum number of tokens for each report
max_n_tokens_train = 0
del_train = []
for i, report in enumerate(annotated_reports['train']):
    x = tokenizer(report.report_text, return_tensors='pt')['input_ids'].shape[1]
    max_n_tokens_train = max(max_n_tokens_train, x)
    if x > model.config.max_position_embeddings:
        del_train.append(i)
print(del_train)
print(f'{max_n_tokens_train = }')

# Check the maximum number of tokens for each report
max_n_tokens_validation = 0
del_val = []
for i, report in enumerate(annotated_reports['validation']):
    x = tokenizer(report.report_text, return_tensors='pt')['input_ids'].shape[1]
    max_n_tokens_validation = max(max_n_tokens_validation, x)
    if x > model.config.max_position_embeddings:
        del_val.append(i)
print(del_val)
print(f'{max_n_tokens_validation = }')

# Delete long reports
for i in del_train[::-1]:
    annotated_reports['train'].pop(i)
for i in del_val[::-1]:
    annotated_reports['validation'].pop(i)
print('deleted')

Token indices sequence length is longer than the specified maximum sequence length for this model (659 > 512). Running this sequence through the model will result in indexing errors


[86, 95, 97, 111]
max_n_tokens_train = 659
[1, 5, 17, 26]
max_n_tokens_validation = 949
deleted


In [10]:
def create_hugging_face_dataset(annotated_reports: list[AnnotatedReport]) -> Dataset:
    text = []
    for report in annotated_reports:
        text.append(report.report_text)
    return Dataset.from_dict({'text': text})

In [11]:
dataset = DatasetDict({
    'train': create_hugging_face_dataset(annotated_reports['train']),
    'validation': create_hugging_face_dataset(annotated_reports['validation'])
})

In [12]:
def tokenize_function(examples, padding="max_length", max_length=model.config.max_position_embeddings):
    return tokenizer(examples['text'], padding=padding, max_length=max_length)

In [13]:
dataset = dataset.map(tokenize_function, batched=True)
dataset.set_format('torch')
print(dataset)

Map:   0%|          | 0/112 [00:00<?, ? examples/s]

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 112
    })
    validation: Dataset({
        features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 24
    })
})


**Test without attention**

In [14]:
input = dataset['train'][0:2]['input_ids'].to(device)
print(input)

tensor([[   101,  10294, 106687,  ...,      0,      0,      0],
        [   101,  10106, 109822,  ...,      0,      0,      0]],
       device='cuda:0')


In [15]:
output = model(input)
print(output['pooler_output'])

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


tensor([[ 0.2992, -0.1450,  0.2331,  ..., -0.3839,  0.2487,  0.2114],
        [ 0.2205, -0.1004,  0.1621,  ..., -0.2607,  0.1066,  0.1246]],
       device='cuda:0', grad_fn=<TanhBackward0>)


**Test with attention**

In [16]:
input_attention = dataset['train'][0:2]['attention_mask'].to(device)
print(input_attention)

tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')


In [17]:
output_attention = model(input, attention_mask=input_attention)
print(output_attention['pooler_output'])

tensor([[ 0.2265, -0.0852,  0.1328,  ..., -0.3184,  0.1100,  0.1947],
        [-0.0737, -0.3641,  0.1564,  ..., -0.4149,  0.1317,  0.2486]],
       device='cuda:0', grad_fn=<TanhBackward0>)


**Test with original text no padding (-> no attention needed)**

Results are the same of those obtained with padding + attention

In [42]:
print(model(torch.tensor(tokenizer(dataset['train'][0:1]['text'])['input_ids']).to(device))['pooler_output'][0][:5])
print(model(torch.tensor(tokenizer(dataset['train'][1:2]['text'])['input_ids']).to(device))['pooler_output'][0][:5])

tensor([ 0.2266, -0.0852,  0.1328, -0.1539, -0.0256], device='cuda:0',
       grad_fn=<SliceBackward0>)
tensor([-0.0736, -0.3641,  0.1564, -0.4477, -0.3163], device='cuda:0',
       grad_fn=<SliceBackward0>)


In [56]:
output_attention['last_hidden_state'][0, 0, :]

tensor([ 1.8137e-01, -1.8808e-01,  2.7782e-01,  4.5238e-02,  1.5821e-01,
         4.2317e-01, -2.3532e-02, -2.4559e-01, -1.5091e-01, -3.3694e-02,
         2.0017e-01,  9.9810e-02, -1.6413e-01,  2.3358e-01, -7.7444e-01,
        -1.1538e-01, -1.3293e-01,  3.1343e-01,  2.3482e-01,  5.2146e-02,
         6.0189e-03, -6.9426e-02, -4.9839e-02, -1.0101e-02, -1.6127e-01,
        -2.0877e-01, -3.2403e-01,  3.4615e-02,  2.8940e-01, -3.8759e-02,
        -3.0309e-02, -1.5973e-01, -2.0700e-01, -4.6922e-02,  2.2681e-01,
        -1.5845e-02, -1.7618e+00, -1.3225e-02, -7.8246e-02, -1.6221e-01,
         1.9012e-01,  3.7651e-01, -1.6619e-01, -1.8195e-01,  1.9962e-02,
         1.0401e+00, -9.1113e-02,  8.7971e-02,  8.0440e-01, -5.0325e-02,
         2.7454e-01, -4.9939e-01, -4.9906e-02, -1.0083e+00, -1.5181e-01,
         2.4927e-02, -1.0427e-01, -1.7372e-01, -4.2001e-02,  2.9274e-02,
         6.6853e-02, -4.0103e-02,  2.2331e-02,  6.1865e-02,  3.5945e-02,
        -4.8023e-02,  1.6421e-01,  1.4531e-01,  1.3

# Test model to predict only morfologia and infiltrazione tessuto adiposo

In [59]:
num_classes_morfologia = 4
num_classes_tessuto_adiposo = 5


class ReportExtractor(nn.Module):
    def __init__(self, bert_checkpoint=CHECKPOINT, num_classes_morfologia=4, num_classes_tessuto_adiposo=5):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_checkpoint)
        hidden = self.bert.config.hidden_size  # Embedding length
        self.dropout = nn.Dropout(0.2)
        # Classification heads
        # Morfologia head
        self.class_morfologia = nn.Linear(hidden, num_classes_morfologia)
        # Tessuto adiposo head
        self.class_tessuto_adiposo = nn.Linear(hidden, num_classes_tessuto_adiposo)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Better than pooler_output
        cls = out.last_hidden_state[:, 0, :]
        # Alternative:
        #cls = out.pooler_output
        cls = self.dropout(cls)
    
        return {
            "morfologia": self.class_morfologia(cls),
            "tessuto_adiposo": self.class_tessuto_adiposo(cls),
        }

In [77]:
report_extractor = ReportExtractor()
report_extractor.eval()

ReportExtractor(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementw

In [79]:
report_extractor(dataset['train'][0:1]['input_ids'], attention_mask=dataset['train'][0:1]['attention_mask'])

{'morfologia': tensor([[ 0.0694,  0.3370,  0.1610, -0.0683]], grad_fn=<AddmmBackward0>),
 'tessuto_adiposo': tensor([[ 0.1746,  0.2191, -0.0386,  0.2296, -0.4250]],
        grad_fn=<AddmmBackward0>)}

In [80]:
logits = report_extractor(dataset['train'][0:1]['input_ids'], attention_mask=dataset['train'][0:1]['attention_mask'])

In [81]:
torch.softmax(logits['morfologia'], dim=-1)

tensor([[0.2340, 0.3058, 0.2564, 0.2039]], grad_fn=<SoftmaxBackward0>)

In [82]:
logits['morfologia']

tensor([[ 0.0694,  0.3370,  0.1610, -0.0683]], grad_fn=<AddmmBackward0>)

TypeError: exp(): argument 'input' (position 1) must be Tensor, not int